# Plato Input Catalogue (PIC) 2.2.0.1

Work in progress! Use on your own risk..

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
# Default packages
import os
import sys
import glob

# PlatoSim default
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PLATOnium default
from pathlib import Path
from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units as u
from tqdm import tqdm

# PlatoSim libraries
import platosim.plot      as pt
import platosim.utilities as ut
import platosim.starquery as sq
from platosim.pic          import PIC2101
from platosim.matplotlibrc import setup_paper
setup_paper(warning=False)

from IPython.display import display, HTML
display(HTML("<style>.container {width:80% !important; }</style>"))

---
## scvPIC
---

Load new scvPIC:

In [ ]:
ds = pd.read_csv('LOPS2scvPICtarget2.2.0.1_input_v1.csv', names=['gaiaDR3', 'scv', 'mag', 'p'])

### Match source with PIC 2.1.0.1 contaminant table

In [ ]:
path = Path(os.getenv('PLATO_PROJECT_HOME')) / 'inputfiles/data_picsim/LOPS2PIC2.1.0.1-t-fg-c-scv/Data' 
filename = path / 'LOPS2PICcontaminant2.1.0.1-t-fg-c-scv.fits'
hdul = fits.open(filename, memmap=True)

In [ ]:
dc = pd.DataFrame()
campos = {
    'gaiaDR3' : 'StarName',
    'ra'      : 'RAdeg', 
    'dec'     : 'DEdeg',
    'pmra'    : 'pmRA',
    'pmdec'   : 'pmDE',
    'Pmag'    : 'PlatoMagNCAM',
    'PBmag'   : 'PlatoMagFCAMb',
    'PRmag'   : 'PlatoMagFCAMr'
}
for campo_dc, campo_pic in campos.items():
    # Print each column to follow progress
    print(campo_pic)
    col = hdul[1].data[campo_pic]
    # Fetch magnitude column and get bool indices for P < 19
    if campo_dc == 'Pmag':
        dx = pd.DataFrame({'Pmag': col})
        dex = dx.Pmag < 17
    # Fetch each column
    dc[campo_dc] = col.byteswap().view(col.dtype.newbyteorder())
print(f'Number of sources found: {dc.shape[0]/1e6}')
dc.head()

Match catalogues with Gaia DR3 entries:

In [ ]:
dc_scv = dc[dc['gaiaDR3'].isin(dt['gaiaDR3'])]
dc_scv.shape[0]

In [ ]:
dt_scv = dt[dt['gaiaDR3'].isin(dc_scv['gaiaDR3'])]
dt_scv.shape[0]

This is the number of stars without Gaia DR3 entries:

In [ ]:
ds.shape[0] - dt_scv.shape[0]

### Handle duplicates sources of the scvPIC

Fetch coordinates and magnitude:

In [ ]:
N = dt_scv.shape[0]
ra = np.zeros(N)
dec = np.zeros(N)
Pmag = np.zeros(N)
for i in tqdm(range(N), bar_format=ut.tqdmBar()):
    dx = dc_scv[dc_scv.gaiaDR3 == dt_scv.gaiaDR3.iloc[i]]
    ra[i] = dx.ra
    dec[i] = dx.dec
    Pmag[i] = dx.Pmag

In [ ]:
df = dt_scv
df['ra']   = ra
df['dec']  = dec
df['Pmag'] = Pmag
gal = SkyCoord(df.ra, df.dec, frame='icrs', unit=u.deg).galactic
df['l'] = gal.l.deg
df['b'] = gal.b.deg

### Handle bright SCV2b stars with different star names

Out of which some are bright stars with different names:

In [ ]:
dx = dt[~dt.gaiaDR3.str.startswith(tuple(['Gaia DR3']))]
dx

In [ ]:
dx.gaiaDR3.iloc[0]

In [ ]:
dx0 = pd.DataFrame()
N = dx.shape[0]
dex = []
for i in tqdm(range(N), bar_format=ut.tqdmBar()):
    try:
        dx_i = dx.iloc[i]
        dx1 = sq.simbadQuery(dx_i.gaiaDR3, radius=100)
        dx1 = dx1.loc[0].to_frame().T
        dx1['']
        dx0 = pd.concat([dx0, dx1])
    except:
        dex.append(i)

In [ ]:
dx0

## scvPIC 2.2.0.2

In [ ]:
dx = pd.read_csv('LOPS2scvPICtarget2.2.0.2.csv')
dx.columns

In [ ]:
# Add Galactic coordinates
gal = SkyCoord(dx.RAdeg, dx.DEdeg, frame='icrs', unit=u.deg).galactic
# Create data frame
df = pd.DataFrame()
df['PIC']     = dx.scvPICid
df['gaiaDR3'] = dx.StarName
df['l']       = gal.l.deg
df['b']       = gal.b.deg
df['ra']      = dx.RAdeg
df['dec']     = dx.DEdeg
df['pmra']    = dx.pmRA
df['pmdec']   = dx.pmDE
df['Pmag']    = dx.PlatoMagNCAM
# dt['PBmag']   = df.PlatoMagFCAMb
# dt['PRmag']   = df.PlatoMagFCAMr
# dt['M']       = df.Mass
# dt['R']       = df.Radius
# dt['Teff']    = df.Teff
# dt['logg']    = pic.get_logg()
df['ncams']   = dx.BOLnCameraObsNCAM_T
# dt['case']    = df.caseFlag
# df['source']  = dx.PICmainSourceFlagBOL
# dt['tPIC']    = df.tPICsourceFlagNCAM_BOL
# dt['fgPIC']   = df.fgPICsourceFlag
# dt['cPIC']    = df.cPICsourceFlag
df['scvPIC']  = dx.scvPICsourceFlag

## Plot each scvPIC sample

In [ ]:
dt = pd.read_feather(path / '../../PIC210_LOPS2_targets.ftr')

In [ ]:
dx = df[(df.ncams != 0)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot P1 sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=10, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} scvPIC targets'
)
fig.savefig('skymap_LOPS2_scvPIC2202.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '1a']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 1 == 1]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot P1 sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=50, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1a targets (eclipsing binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV1a.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '1b']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 2 == 2]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot P1 sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=50, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1b targets (astrometric binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV1b.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '1c']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 4 == 4]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot P1 sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1c targets (wide binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV1c.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '1d']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 8 == 8]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot P1 sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=200, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1d targets (HW Vir-type binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV1d.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '1e']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 16 == 16]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot P1 sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1e targets (wide WD binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV1e.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '2a']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 32 == 32]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot P1 sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV2a targets (legacy and benchmark)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV2a.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '2b']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 64 == 64]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=32, ncamStars=dt,
    # Plot P1 sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=50, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV2b targets (legacy and benchmark)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV2b.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '3a']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 128 == 128]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV3a targets (photometric stable)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV3a.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '3b']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 256 == 256]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV3b targets (photometric stable)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV3b.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '4a']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 512 == 512]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV4a targets (solar-like pulsators)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV4a.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '4b']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 1024 == 1024]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=10, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV4b targets (solar-like pulsators)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV4b.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '5']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 2048 == 2048]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=30, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV5 targets (gamma Dor pulsators)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV5.png', bbox_inches='tight', dpi=200)

In [ ]:
# dx = df[df.scv == '6']
dx = df[(df.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 4096 == 4096]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot sample stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=200, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV6 targets (transiting exoplanets)'
)
fig.savefig('skymap_LOPS2_scvPIC2202_SCV6.png', bbox_inches='tight', dpi=200)